In [ ]:
# Install
!pip install tensorflow
!pip install transformers datasets torch

# Imports
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense
from transformers import pipeline

In [ ]:
vocab_size = 10000
max_len = 200

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)

word_index = imdb.get_word_index()

x_train = pad_sequences(x_train, maxlen=max_len)
x_test = pad_sequences(x_test, maxlen=max_len)

model = Sequential([
    Embedding(vocab_size, 64, input_length=max_len),
    SimpleRNN(64),
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

model.fit(x_train, y_train, epochs=2, batch_size=64)

def encode_text(text):
    tokens = text.lower().split()
    encoded = [word_index.get(word, 2) for word in tokens]  # 2 = unknown
    return pad_sequences([encoded], maxlen=max_len)

sample = "this movie was amazing and beautiful"
encoded = encode_text(sample)

prediction = model.predict(encoded)[0][0]

print(sample)
print("Sentiment:", "Positive" if prediction > 0.5 else "Negative")

Epoch 1/2
391/391 ━━━━━━━━━━━━━━━━━━━━ 31s 73ms/step - accuracy: 0.6650 - loss: 0.5963
Epoch 2/2
391/391 ━━━━━━━━━━━━━━━━━━━━ 29s 73ms/step - accuracy: 0.8170 - loss: 0.4233
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step
this movie was amazing and beautiful
Sentiment: Positive


In [ ]:
model = Sequential([
    Embedding(vocab_size, 64, input_length=max_len),
    LSTM(64),
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

model.fit(x_train, y_train, epochs=2, batch_size=64)

def encode_text(text):
    tokens = text.lower().split()
    encoded = [word_index.get(word, 2) for word in tokens]
    return pad_sequences([encoded], maxlen=max_len)

sample = "this movie was terrible and boring"
encoded = encode_text(sample)

prediction = model.predict(encoded)[0][0]

print(sample)
print("Sentiment:", "Negative" if prediction > 0.5 else "Positive")

Epoch 1/2
391/391 ━━━━━━━━━━━━━━━━━━━━ 65s 160ms/step - accuracy: 0.7935 - loss: 0.4348
Epoch 2/2
391/391 ━━━━━━━━━━━━━━━━━━━━ 82s 161ms/step - accuracy: 0.8970 - loss: 0.2596


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 225ms/step
this movie was terrible and boring
Sentiment: Negative


In [ ]:
classifier = pipeline("sentiment-analysis")

texts = [
    "I love this movie!",
    "This was the worst thing ever",
    "Honestly kinda mid"
]

results = classifier(texts)

for text, res in zip(texts, results):
    print(text, "->", res)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

I love this movie! -> {'label': 'POSITIVE', 'score': 0.9998775720596313}
This was the worst thing ever -> {'label': 'NEGATIVE', 'score': 0.9997575879096985}
Honestly kinda mid -> {'label': 'NEGATIVE', 'score': 0.9887430667877197}


In [ ]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification
import torch
import torch.nn.functional as F


model_name = "cardiffnlp/twitter-roberta-base-sentiment"

tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaForSequenceClassification.from_pretrained(model_name)

model.eval()

labels = ["Negative", "Neutral", "Positive"]

def predict_sentiment(text):

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)


    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits


    probs = F.softmax(logits, dim=1)

    predicted_class = torch.argmax(probs, dim=1).item()

    return labels[predicted_class], probs[0].tolist()

texts = [
    "this movie was amazing and beautiful",
    "this movie was terrible and boring",
    "it was okay, not great",
    "average experience nothing special"
]

for text in texts:
    label, prob = predict_sentiment(text)

    print("Text:", text)
    print("Sentiment:", label)
    print("Probabilities [Neg, Neu, Pos]:", prob)
    print()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Text: this movie was amazing and beautiful
Sentiment: Positive
Probabilities [Neg, Neu, Pos]: [0.002189710270613432, 0.008655233308672905, 0.989155113697052]

Text: this movie was terrible and boring
Sentiment: Negative
Probabilities [Neg, Neu, Pos]: [0.9818426370620728, 0.015515225008130074, 0.0026421574875712395]

Text: it was okay, not great
Sentiment: Negative
Probabilities [Neg, Neu, Pos]: [0.53082275390625, 0.3746241629123688, 0.09455306828022003]

Text: average experience nothing special
Sentiment: Negative
Probabilities [Neg, Neu, Pos]: [0.5655593276023865, 0.3963926136493683, 0.03804808109998703]

